In [1]:
import geopandas as gpd
from shapely.ops import snap
import networkx as nx
from shapely.geometry import Point
import numpy as np
from pysheds.grid import Grid
import pandas as pd

In [2]:
watershed = gpd.read_file("GIS/WATERSHEDS_CLIPPED.geojson")
watershed = watershed[watershed['NAME'] == 'Popes Head Creek']
watershed = watershed.to_crs('EPSG:26918')

In [3]:
grid = Grid.from_raster("elevation/popes_head_dem.tif")
dem = grid.read_raster("elevation/popes_head_dem.tif")

In [4]:
s14 = gpd.read_file('stormnet/14.geojson')
s14 = s14.to_crs(watershed.crs).clip(watershed)

# s14['YEAR_BUILT'] = pd.to_datetime(s14.CREATION_DATE, unit='ms').dt.year
# s14['YEAR_BUILT'] = s14['YEAR_BUILT'].fillna(s14.YEAR_BUILT.median()).astype(int)

In [5]:
def get_index(x, y):
    j, i = ~grid.affine * (x, y)
    return int(i), int(j)

G = nx.DiGraph()
for idx, row in s14.iterrows():
    if row.geometry.geom_type != 'LineString':
        continue
    line = row.geometry
    
    start_point = Point(line.coords[0])
    end_point = Point(line.coords[-1])
    
    start_id = (round(start_point.x, 3), round(start_point.y, 3))
    end_id = (round(end_point.x, 3),round(end_point.y, 3))

    G.add_node(start_id, x=start_point.x, y=start_point.y)
    G.add_node(end_id, x=end_point.x,   y=end_point.y)

    G.add_edge(
        start_id,
        end_id,
        pipe_id=row["STORMNET_ID"],
        start_node=start_id,
        end_node=end_id,
        # length=row["ASSETLENGTH"],
        # slope=row["SLOPE"],
        slope=None, # DERIVE THIS
        diameter=row["HEIGHT_DIA"],
        material=row["MATERIAL"],
        geometry=line,
        # year_built=row['YEAR_BUILT']
    )


In [6]:
for node in list(G.nodes):
    if G.out_degree(node) == 0 and G.in_degree(node) > 1:
        preds = list(G.predecessors(node))
        keeper = preds[0]
        for p in preds[1:]: # artificially reroute from outlet to junction for SWMM
            data = G.get_edge_data(p, node)
            data['end_node'] = keeper
            G.remove_edge(p, node)
            G.add_edge(
                p, keeper,
                **data
            )
    elif G.out_degree(node) == 0 and G.in_degree(node) == 0:
        G.remove_node(node)

In [7]:
infalls = []
outfalls = []

for node in G.nodes:
    x, y = G.nodes[node]['x'], G.nodes[node]['y']
    j, i = ~grid.affine * (x, y)
    i, j = int(i), int(j)
    elevation = dem[i, j]
    if elevation == dem.nodata:
        elevation = None
    G.nodes[node]['elevation'] = elevation
    if G.out_degree(node) == 0:
        outfalls.append(node)
        G.nodes[node]['node_type'] = 'outfall'
    elif G.in_degree(node) == 0:
        infalls.append(node)
        G.nodes[node]['node_type'] = 'infall'
    else:
        G.nodes[node]['node_type'] = 'junction'

print(f"Number of infalls: {len(infalls)}")
print(f"Number of outfalls: {len(outfalls)}")


Number of infalls: 1389
Number of outfalls: 961


In [8]:
node_geoms = []
node_records = []

for n, data in G.nodes(data=True):
    rec = {k: v for k, v in data.items() if k != "x" and k != "y"}
    rec["node_id"] = n
    node_records.append(rec)
    node_geoms.append(Point(data["x"], data["y"]))

nodes_gdf = gpd.GeoDataFrame(
    node_records,
    geometry=node_geoms,
    crs=s14.crs
)
median_elevation = nodes_gdf.elevation.median() # fill nan
nodes_gdf.elevation = nodes_gdf.elevation.fillna(median_elevation)

edge_geoms = []
edge_records = []

for u, v, data in G.edges(data=True):
    geom = data["geometry"]
    rec = {k: v for k, v in data.items() if k != "geometry"}
    
    edge_geoms.append(geom)
    edge_records.append(rec)

edges_gdf = gpd.GeoDataFrame(edge_records, geometry=edge_geoms, crs=s14.crs)

for p0, p1, data in G.edges(data=True):
    p0_elev = G.nodes[p0]['elevation']
    p1_elev = G.nodes[p1]['elevation']
    if not p0_elev or not p1_elev:
        edges_gdf['slope'] = 0.0
        continue
    edges_gdf['slope'] = (p0_elev - p1_elev)
edges_gdf['slope'] = (edges_gdf.slope / edges_gdf.length).clip(0.00, 0.1)*100
edges_gdf['diameter'] = (edges_gdf.diameter.fillna(edges_gdf.diameter.median()) / 39.37) # convert inches to meters
edges_gdf.loc[edges_gdf.diameter < 0, "diameter"] =  edges_gdf.diameter[edges_gdf.diameter > 0].median()

In [9]:
nodes_gdf.to_file("nodes.geojson", driver="GeoJSON")
edges_gdf.to_file("edges.geojson", driver="GeoJSON")